# Phase 1 — Test: lib/ Skeleton + show_source + should_skip (v2)

**v2:** `show_source` rendert jetzt standardmässig als **Markdown** —
JupyterLab nutzt sein eigenes Syntax-Highlighting, das zum aktiven Theme passt.
Kein Kontrastproblem mehr bei hellem Theme.

Dieses Notebook prüft, dass:

1. Das `lib/`-Package korrekt importierbar ist
2. `show_source(...)` im **Markdown-Modus** aufklappbare Quellcode-Blöcke erzeugt (default)
3. `show_source(..., mode='html', style=...)` alternativ pygments nutzt
4. `should_skip(...)` korrekt entscheidet

**Voraussetzung:** Dieses Notebook liegt in `test/`, also eine Ebene unter dem Projekt-Root.
Falls du es auf Projekt-Root-Ebene legst, ändere unten `_lib_root = Path('.').resolve()`.

In [14]:
# ── lib-Import Setup ─────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Dieses Notebook liegt in test/ — ein Ebene hoch zu Projekt-Root:
_lib_root = Path('..').resolve()
if str(_lib_root) not in sys.path:
    sys.path.insert(0, str(_lib_root))

%load_ext autoreload
%autoreload 2

print(f'sys.path enthält: {_lib_root}')
print(f'lib/ existiert:   {(_lib_root / "lib").exists()}')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
sys.path enthält: C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage
lib/ existiert:   True


## Test 1 — Imports funktionieren


In [15]:
from lib.plotting import show_source, should_skip
from lib import __version__ as lib_version

print(f'✅ lib/ Version {lib_version} geladen')
print(f'   show_source:  {show_source}')
print(f'   should_skip:  {should_skip}')


✅ lib/ Version 0.1.0 geladen
   show_source:  <function show_source at 0x000001E9FC6DBAC0>
   should_skip:  <function should_skip at 0x000001E9FC6DBC70>


## Test 2 — `show_source()` Default-Modus (Markdown)

Unter dieser Zelle sollte ein **zugeklappter** Block erscheinen:  
🔎 *Quellcode: `should_skip` (aus `lib/plotting.py`)*  

Beim Klick öffnet sich ein Codeblock mit JupyterLab's nativem Syntax-
Highlighting — passt automatisch zum aktiven Theme (hell/dunkel).

In [16]:
show_source(should_skip)


<details>
<summary>🔎 Quellcode: <code>should_skip</code> (aus <code>lib/plotting.py</code>)</summary>

```python
def should_skip(
    out_path: str,
    asset_type: str,
    name: str,
    cfg: dict[str, Any],
) -> bool:
    """Prüft anhand der Config, ob ein Chart/GIF neu erzeugt werden muss.

    Wird in jeder Zelle genutzt, die ein rechenintensives Artefakt erzeugt.
    Schema der relevanten Config-Sektion (in ``sync/config.json``):

    .. code-block:: json

        "animation": {
            "modus":          "skip_if_exists",
            "modus_statisch": "always",
            "overrides": {
                "kuer_k01_karte_verbraucher": "skip_if_exists"
            }
        }

    Entscheidungsreihenfolge
    ------------------------
    1. ``overrides[name]`` — explizit gesetzter Wert (höchste Priorität)
    2. ``asset_type == 'animation'`` → ``cfg['animation']['modus']``
       (Default: ``'skip_if_exists'``)
    3. ``asset_type == 'statisch'`` → ``cfg['animation']['modus_statisch']``
       (Default: ``'always'``). Wert ``'skip_if_exists_all'`` schaltet
       alle statischen Charts auf skip_if_exists um (Master-Schalter).

    Mode-Werte
    ----------
    * ``'skip_if_exists'``  — skip wenn Datei existiert und nicht leer
    * ``'always'``          — nie skip (Datei wird immer überschrieben)
    * ``'force_rebuild'``   — Alias für ``'always'``

    Parameter
    ---------
    out_path : str
        Zielpfad des zu erzeugenden Artefakts.
    asset_type : str
        Entweder ``'animation'`` oder ``'statisch'``.
    name : str
        Basename des Artefakts (ohne Extension), für ``overrides``-Lookup.
        Beispiel: ``'kuer_k04_anim_A_18h'``.
    cfg : dict
        Geladenes CFG-Dict aus ``sync/config.json``.

    Rückgabe
    --------
    bool
        ``True`` wenn die Erzeugung übersprungen werden kann, sonst ``False``.

    Beispiel
    --------
    >>> out = os.path.join(CHARTS_DIR, 'kuer_k04_anim_A_18h.gif')
    >>> if should_skip(out, 'animation', 'kuer_k04_anim_A_18h', CFG):
    ...     print(f"✓ Überspringe (existiert)")
    ... else:
    ...     make_gif_chart(...)
    """
    if asset_type not in ("animation", "statisch"):
        raise ValueError(
            f"asset_type muss 'animation' oder 'statisch' sein, "
            f"nicht {asset_type!r}"
        )

    anim_cfg = cfg.get("animation", {})
    overrides = anim_cfg.get("overrides", {})

    # 1. Explizit gesetzter Override (Keys mit _ sind Hinweis-Felder)
    if name in overrides and not name.startswith("_"):
        mode = overrides[name]
    # 2. Global je Asset-Typ
    elif asset_type == "animation":
        mode = anim_cfg.get("modus", "skip_if_exists")
    else:  # 'statisch'
        global_static = anim_cfg.get("modus_statisch", "always")
        # Master-Schalter: schaltet ALLE statischen Charts auf skip_if_exists
        if global_static == "skip_if_exists_all":
            mode = "skip_if_exists"
        else:
            mode = global_static

    # Validierung des Mode-Werts (nur informativ, fehlerhafte Werte → 'always')
    valid_modes = {"skip_if_exists", "always", "force_rebuild"}
    if mode not in valid_modes:
        # Nicht hart abbrechen — SSOT-Violation soll loud, aber nicht blockierend
        # sein, damit der Notebook-Lauf nicht an einem Tippfehler hängt.
        import warnings
        warnings.warn(
            f"Unbekannter Modus {mode!r} für {asset_type} '{name}'. "
            f"Erlaubt: {sorted(valid_modes)}. Fallback auf 'always'.",
            RuntimeWarning,
        )
        mode = "always"

    if mode == "skip_if_exists":
        return os.path.exists(out_path) and os.path.getsize(out_path) > 0
    # 'always' oder 'force_rebuild' → immer neu erzeugen
    return False
```

</details>


## Test 3 — `collapsed=False` — offen statt aufgeklappt


In [5]:
show_source(show_source, collapsed=False)


**🔎 Quellcode: `show_source` (aus `lib/plotting.py`)**

```python
def show_source(
    func: Callable,
    title: str | None = None,
    collapsed: bool = True,
    mode: str = "markdown",
    style: str = "default",
) -> None:
    """Zeigt den Quellcode einer Funktion im Notebook — theme-passend oder als HTML.

    Wird in Notebooks unmittelbar nach ``from lib... import`` verwendet, damit
    Dozenten/Reviewer den Code direkt inline sehen ohne die ``.py`` öffnen zu
    müssen. Der Quellcode wird zur Render-Zeit via ``inspect.getsource`` geholt
    — bei jedem Re-Run der Zelle wird die *aktuelle* lib-Version gezeigt.

    Zwei Rendering-Modi:

    * ``mode='markdown'`` (Default): Nutzt ``IPython.display.Markdown``.
      JupyterLab rendert den Codeblock mit seinem **eigenen** Syntax-
      Highlighting, das zum aktiven Theme (hell/dunkel) passt. Kein
      Kontrastproblem. Klappbar via ``<details>`` (funktioniert in
      JupyterLab 4+).

    * ``mode='html'``: Nutzt ``IPython.display.HTML`` mit pygments-Rendering.
      Fester Style (``style``-Parameter), unabhängig vom Notebook-Theme.
      Nützlich wenn Markdown-Rendering nicht zuverlässig ist (z.B. nbviewer
      in älteren Versionen, Export als HTML-Report).

    Parameter
    ---------
    func : Callable
        Die Funktion (oder Klasse), deren Quellcode angezeigt werden soll.
    title : str, optional
        Überschreibt den automatisch generierten Titel
        ``Quellcode: <n> (aus <modul>.py)``.
    collapsed : bool, default True
        Ob die Anzeige initial zugeklappt (``<details>``) oder offen sein soll.
    mode : {'markdown', 'html'}, default 'markdown'
        Rendering-Modus.
    style : str, default 'default'
        Nur wirksam bei ``mode='html'``. Pygments-Style-Name. Empfehlungen:

        * ``'default'``  — hell, guter Kontrast (weisser HG)
        * ``'friendly'`` — hell, freundlich
        * ``'tango'``    — hell, kräftige Farben
        * ``'monokai'``  — dunkel, hoher Kontrast
        * ``'github-dark'`` — dunkel, GitHub-Stil (weniger Kontrast bei Kommentaren)
        * ``'one-dark'`` — dunkel, VS-Code-Stil

        Liste aller verfügbaren:
        ``from pygments.styles import get_all_styles; list(get_all_styles())``

    Beispiele
    --------
    >>> from lib.simulation import simulate_battery
    >>> from lib.plotting import show_source
    >>> show_source(simulate_battery)                         # Markdown, theme-passend
    >>> show_source(simulate_battery, collapsed=False)        # offen
    >>> show_source(simulate_battery, mode='html', style='monokai')  # HTML pygments
    """
    import inspect
    from IPython.display import display

    # Quellcode holen — robust gegen nicht-introspizierbare Objekte
    try:
        src = inspect.getsource(func)
    except (TypeError, OSError) as e:
        from IPython.display import Markdown
        name = getattr(func, "__name__", repr(func))
        display(Markdown(
            f"> ⚠️ Quellcode für `{name}` nicht verfügbar: {e}"
        ))
        return

    # Titel generieren
    mod = getattr(func, "__module__", "?")
    mod_path = mod.replace(".", "/")
    name = getattr(func, "__name__", "?")
    caption = title or f"Quellcode: `{name}` (aus `{mod_path}.py`)"

    if mode == "markdown":
        _show_source_markdown(src, caption, collapsed)
    elif mode == "html":
        _show_source_html(src, caption, collapsed, style)
    else:
        raise ValueError(
            f"mode muss 'markdown' oder 'html' sein, nicht {mode!r}"
        )
```


## Test 4 — HTML-Fallback mit verschiedenen pygments-Styles

Falls das Markdown-Rendering in deinem Setup nicht funktioniert (ältere
nbviewer-Versionen, Moodle-Preview, HTML-Export), kannst du `mode='html'`
nutzen. Dann kommt pygments mit fest gewähltem Style zum Einsatz.

Verfügbare Style-Empfehlungen:

- **Hell:** `default`, `friendly`, `tango`, `sas`
- **Dunkel:** `monokai`, `one-dark`, `github-dark`, `dracula` (falls installiert)

Vergleiche unten die Lesbarkeit:

In [6]:
# Hell: 'default' (sehr lesbar auf weissem Jupyter-HG)
show_source(should_skip, mode='html', style='default',
            title='HTML + pygments style="default" (hell)')


In [7]:
# Hell: 'friendly' — weicher, pastell-artig
show_source(should_skip, mode='html', style='friendly',
            title='HTML + pygments style="friendly" (hell, weich)')


In [8]:
# Dunkel: 'monokai' — klassisch, hoher Kontrast
show_source(should_skip, mode='html', style='monokai',
            title='HTML + pygments style="monokai" (dunkel, hoher Kontrast)')


In [9]:
# Dunkel: 'one-dark' — wie VS Code
show_source(should_skip, mode='html', style='one-dark',
            title='HTML + pygments style="one-dark" (VS Code)')


### Alle verfügbaren pygments-Styles auflisten


In [10]:
from pygments.styles import get_all_styles
print('Verfügbare pygments-Styles:')
for s in sorted(get_all_styles()):
    print(f'  {s}')


Verfügbare pygments-Styles:
  abap
  algol
  algol_nu
  arduino
  autumn
  borland
  bw
  coffee
  colorful
  default
  dracula
  emacs
  friendly
  friendly_grayscale
  fruity
  github-dark
  gruvbox-dark
  gruvbox-light
  igor
  inkpot
  lightbulb
  lilypond
  lovelace
  manni
  material
  monokai
  murphy
  native
  nord
  nord-darker
  one-dark
  paraiso-dark
  paraiso-light
  pastie
  perldoc
  rainbow_dash
  rrt
  sas
  solarized-dark
  solarized-light
  staroffice
  stata-dark
  stata-light
  tango
  trac
  vim
  vs
  xcode
  zenburn


## Test 5 — `should_skip()` Entscheidungs-Matrix

Prüfung aller relevanten Pfade:

| Fall | out_path | asset_type | name | overrides | Erwartung |
|---|---|---|---|---|---|
| 1 | existiert | animation | — | — | `True` (skip) |
| 2 | fehlt | animation | — | — | `False` (rendern) |
| 3 | existiert | statisch | — | — | `False` (default always) |
| 4 | existiert | statisch | match override skip | skip | `True` (skip) |
| 5 | existiert | animation | match override force | force | `False` (rendern) |


In [11]:
import os, tempfile

_tmp = tempfile.mkdtemp()
_exists = os.path.join(_tmp, 'exists.gif')
_missing = os.path.join(_tmp, 'missing.gif')
open(_exists, 'w').write('dummy')

CFG = {
    'animation': {
        'modus':          'skip_if_exists',
        'modus_statisch': 'always',
        'overrides': {
            'my_expensive_map': 'skip_if_exists',
            'my_dev_chart':     'force_rebuild',
        }
    }
}

tests = [
    (_exists,  'animation', 'some_anim',         True,  'Fall 1: anim exists, default skip'),
    (_missing, 'animation', 'some_anim',         False, 'Fall 2: anim fehlt'),
    (_exists,  'statisch',  'some_chart',        False, 'Fall 3: stat exists, default always'),
    (_exists,  'statisch',  'my_expensive_map',  True,  'Fall 4: stat exists, override skip'),
    (_exists,  'animation', 'my_dev_chart',      False, 'Fall 5: anim exists, override force'),
]

for out, typ, name, expected, desc in tests:
    got = should_skip(out, typ, name, CFG)
    status = '✅' if got == expected else '❌'
    print(f'{status} {desc:<40s} got={got} expected={expected}')


✅ Fall 1: anim exists, default skip        got=True expected=True
✅ Fall 2: anim fehlt                       got=False expected=False
✅ Fall 3: stat exists, default always      got=False expected=False
✅ Fall 4: stat exists, override skip       got=True expected=True
✅ Fall 5: anim exists, override force      got=False expected=False


## Test 6 — Master-Schalter `skip_if_exists_all`

Setzt man `CFG['animation']['modus_statisch'] = 'skip_if_exists_all'`, werden
**alle** statischen Charts auf `skip_if_exists` umgeschaltet.

In [12]:
CFG_master = {
    'animation': {
        'modus': 'skip_if_exists',
        'modus_statisch': 'skip_if_exists_all',
    }
}

got = should_skip(_exists, 'statisch', 'any_chart_name', CFG_master)
print(f'{"✅" if got else "❌"} Master-Schalter skip_if_exists_all → {got}')


✅ Master-Schalter skip_if_exists_all → True


## Test 7 — Robustheit gegen fehlerhafte Inputs


In [13]:
import warnings

# Falsch getippter Modus → Warning + Fallback
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    got = should_skip(_exists, 'animation', 'x', {'animation': {'modus': 'typo'}})
    print(f'{"✅" if got == False and len(w) == 1 else "❌"} Typo im Modus → {len(w)} Warning(s), Ergebnis {got}')

# asset_type falsch → ValueError
try:
    should_skip(_exists, 'video', 'x', CFG)
    print('❌ Hätte ValueError werfen sollen')
except ValueError as e:
    print(f'✅ ValueError bei asset_type=video: {e}')

# show_source mit ungültigem mode → ValueError
try:
    show_source(should_skip, mode='xml')
    print('❌ Hätte ValueError werfen sollen')
except ValueError as e:
    print(f'✅ ValueError bei mode=xml: {e}')


✅ Typo im Modus → 1 Warning(s), Ergebnis False
✅ ValueError bei asset_type=video: asset_type muss 'animation' oder 'statisch' sein, nicht 'video'
✅ ValueError bei mode=xml: mode muss 'markdown' oder 'html' sein, nicht 'xml'


## Abschluss

Wenn alle Tests ✅ zeigen und Test 2 einen aufklappbaren Codeblock mit
korrekt lesbarem Syntax-Highlighting (passend zu deinem JupyterLab-Theme)
produziert, ist **Phase 1** erfolgreich.

### Empfehlung

- **In den Projekt-Notebooks** immer `show_source(fn)` (default markdown) nutzen
  — bleibt theme-passend, bester Kontrast
- **`mode='html'` nur wenn nötig**, z.B. für HTML-Export oder Moodle-Preview
- **Style-Parameter erst relevant** wenn HTML-Mode + Theme-Unabhängigkeit gebraucht werden

### Nächster Schritt

Phase 2 (Anker-Hygiene) — mechanische Bereinigung von ID-Ankern mit
führenden Ziffern in 8 Notebooks.
